# 3.1 — Set Up Model Access Controls

**Exam domain:** Gen AI Governance · **Weight:** 28%

## The problem this solves

Your company turns on Snowflake and, without anyone deciding it, every single user can now send
customer support tickets to a large language model. Finance discovers the bill a month later.
Legal discovers that EU customer data was processed in a US region. Nobody did anything wrong —
the defaults were simply open, and no one closed them.

This notebook is about closing them deliberately: who may call an AI function, which models they
may call, where the inference is allowed to happen, and what the model is allowed to see.

## What you will be able to do

- Revoke the permissive Cortex defaults and grant AI access back, role by role
- Restrict which models a specific role can call, and explain why the account-wide allowlist cannot do that job alone
- Pin inference to a cloud and geography for a data-residency requirement
- Strip PII out of text before it ever reaches a model, and filter unsafe output after
- Test a restricted role honestly, without secondary roles hiding the result

## Before you start

- A role that can run `ALTER ACCOUNT` and `GRANT` — in practice `ACCOUNTADMIN`
- The `GENAI_STUDY` database and its `SUPPORT_TICKETS` table from the repo setup script
- A warehouse you can run queries on

📖 **Snowflake documentation for this notebook**
- [AISQL privileges and model access](https://docs.snowflake.com/en/user-guide/snowflake-cortex/aisql-privileges-and-access)
- [SNOWFLAKE database roles](https://docs.snowflake.com/en/sql-reference/snowflake-db-roles)
- [Cross-region inference](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cross-region-inference)
- [Opting out of Snowflake AI features](https://docs.snowflake.com/en/user-guide/snowflake-cortex/opting-out)
- [AI_REDACT](https://docs.snowflake.com/en/sql-reference/functions/ai_redact)
- [AI_COMPLETE (single string)](https://docs.snowflake.com/en/sql-reference/functions/ai_complete-single-string)


---
## Five gates, in order

An analogy worth holding onto: getting a Cortex call through is like clearing five checkpoints at an
airport. You do not need to be cleared once; you need to be cleared at each gate, and the first one
that turns you back is the error message you see. Everything else in Domain 3 hangs off this order.

```
1. Account privilege   ->  USE AI FUNCTIONS ON ACCOUNT
                           (or a per-function GRANT USE AI FUNCTION <name> ON ACCOUNT)
2. Database role       ->  SNOWFLAKE.CORTEX_USER  or  SNOWFLAKE.AI_FUNCTIONS_USER
                           (feature roles: CORTEX_EMBED_USER, CORTEX_ANALYST_USER,
                            CORTEX_AGENT_USER, CORTEX_REST_API_USER, COPILOT_USER)
3. Model access        ->  model RBAC application role  OR  CORTEX_MODELS_ALLOWLIST
4. Region routing      ->  CORTEX_ENABLED_CROSS_REGION
5. Data access         ->  USAGE on database/schema, SELECT on tables, READ on stages,
                           USAGE on warehouse, plus masking and redaction on the data itself
```

A **privilege** is a permission on the account or an object. A **database role** is a named bundle of
privileges that lives inside a database — the Cortex ones live in the `SNOWFLAKE` database and you
grant them to your own account roles, never to users directly.

| Control | What it does |
|---|---|
| `USE AI FUNCTIONS` | Account privilege required for AI function calls. Granted to `PUBLIC` by default |
| `USE AI FUNCTION <name>` | The same thing for one named function. Combines with the blanket privilege as an OR |
| `SNOWFLAKE.CORTEX_USER` | Covers Cortex AI functions and Cortex services. Granted to `PUBLIC` by default |
| `SNOWFLAKE.AI_FUNCTIONS_USER` | Scalar AI functions only — excludes `AI_AGG` and `AI_SUMMARIZE_AGG` and all Cortex services. Not granted by default |
| Model RBAC application roles | Per-model access: `SNOWFLAKE."CORTEX-MODEL-ROLE-<MODEL>"`, plus `CORTEX-MODEL-ROLE-ALL` |
| `CORTEX_MODELS_ALLOWLIST` | Account-wide list of usable models. Default `'All'` |
| `CORTEX_ENABLED_CROSS_REGION` | Whether inference may leave the home region, and how far |
| Cortex Guard | Output safety filter — `model_parameters => {'guardrails': TRUE}`, default `FALSE` |
| `AI_REDACT` | Removes PII from text before it is processed, stored or shared |

→ [More on AISQL privileges and model access](https://docs.snowflake.com/en/user-guide/snowflake-cortex/aisql-privileges-and-access)
→ [More on the SNOWFLAKE database roles](https://docs.snowflake.com/en/sql-reference/snowflake-db-roles)


---
## Gate 4 first: where is the inference actually happening?

Start here because it is the gate most people never look at. Not every model is served in every
Snowflake region. When a model is missing from your region, Snowflake can route the request to
another region and send the answer back — but only as far as `CORTEX_ENABLED_CROSS_REGION` allows.

Only `ACCOUNTADMIN` can change this parameter, with `ALTER ACCOUNT`.

| Tier | Values |
|---|---|
| Anywhere | `ANY_REGION` |
| One cloud | `AWS_GLOBAL` · `AZURE_GLOBAL` · `GCP_GLOBAL` |
| Cloud + geography | `AWS_US` · `AWS_EU` · `AWS_APJ` · `AWS_JP` · `AWS_AU` · `AZURE_US` · `AZURE_EU` · `GCP_US` |
| Off | `DISABLED` |

Comma-separated combinations of the cloud and cloud+geography values are supported. The default is
`ANY_REGION` for new accounts in new organizations created after 9 March 2026; other commercial
accounts inherit a same-cloud geography value such as `AWS_US` or `AZURE_EU`.

What the docs commit to about the traffic itself: within one cloud provider the data stays on that
provider's private backbone and never crosses the public internet; across providers it crosses the
public internet under mutual TLS. No customer data is stored in the processing region, credits are
consumed in the **requesting** region, and you are not charged data egress.

**The trade-off.** `ANY_REGION` gives you the widest model catalogue and the fewest "model not
available" failures. `AWS_EU` gives you a residency story you can put in front of an auditor, and
costs you every model that is not served in AWS Europe. There is no single-region value — cloud plus
geography is the finest granularity available.

→ [More on cross-region inference](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cross-region-inference)


In [ ]:
%%sql -r xregion_param
-- Cross-region inference: where may a request be served?
SHOW PARAMETERS LIKE 'CORTEX_ENABLED_CROSS_REGION' IN ACCOUNT;


In [ ]:
%%sql
-- Valid values:
--   ANY_REGION
--   AWS_GLOBAL | AZURE_GLOBAL | GCP_GLOBAL                         (one cloud)
--   AWS_US | AWS_EU | AWS_APJ | AWS_JP | AWS_AU
--   AZURE_US | AZURE_EU | GCP_US                                   (cloud + geography)
--   DISABLED
-- Comma-separated combinations of the cloud and cloud+geography values are supported.
ALTER ACCOUNT SET CORTEX_ENABLED_CROSS_REGION = 'AWS_GLOBAL';   -- any AWS region, no Azure or GCP


In [ ]:
%%sql -r models_for_role
-- Only ACCOUNTADMIN may set this parameter.
-- Default is ANY_REGION for new accounts in new organizations created after 2026-03-09;
-- other commercial accounts inherit a same-cloud geography value such as AWS_US or AZURE_EU.

-- Which models can THIS role reach, and where?
SHOW CORTEX BASE MODELS IN SCHEMA SNOWFLAKE.MODELS;
-- Columns: created_on, name, model_type, database_name, schema_name, owner,
--          lifecycle_status, in_region_availability, legacy_date, eol_date,
--          cross_region_availability


---
## Gate 3: which models may this role call?

Two independent mechanisms, and the exam leans on the fact that they combine with **OR**:

```
the role may call model M if
      the role holds SNOWFLAKE."CORTEX-MODEL-ROLE-<M>"   (or CORTEX-MODEL-ROLE-ALL)
   OR M matches an entry in CORTEX_MODELS_ALLOWLIST
```

| | `CORTEX_MODELS_ALLOWLIST` | Model RBAC application roles |
|---|---|---|
| Scope | Account-wide, identical for every role | Per role |
| Values | `'All'` (default), `'None'`, or a comma-separated list of lowercase model names | Grant or revoke one application role per model |
| Set where | `ALTER ACCOUNT SET …` — account level only, not session or user | `GRANT APPLICATION ROLE SNOWFLAKE."CORTEX-MODEL-ROLE-…"` |
| Status | Being retired: from August 2026 the only change still permitted is setting it to `'None'`, and it is removed later in 2026 | The mechanism to build on |

The per-model application roles only exist once the model objects have been created in
`SNOWFLAKE.MODELS`. Snowflake refreshes that daily; `ACCOUNTADMIN` can force it with
`CALL SNOWFLAKE.MODELS.CORTEX_BASE_MODELS_REFRESH();`.

Because the relationship is OR, **the allowlist cannot take a model away from a role that holds the
application role.** Setting `CORTEX_MODELS_ALLOWLIST = 'None'` does not lock the account down on its
own — it switches the account-wide path off and leaves model RBAC in sole charge. That is exactly
what you want when different teams need different models.

→ [More on model access control](https://docs.snowflake.com/en/user-guide/snowflake-cortex/aisql-privileges-and-access)
→ [More on SHOW CORTEX BASE MODELS](https://docs.snowflake.com/en/sql-reference/sql/show-cortex-base-models)


In [ ]:
%%sql
-- ============================================================
-- RESTRICTING WHICH MODELS A ROLE MAY CALL
-- Two mechanisms, combined with OR:
--   (A) CORTEX_MODELS_ALLOWLIST       -> account-wide, identical for every role
--   (B) model RBAC application roles  -> per role; this is how you differentiate teams
-- ============================================================
USE ROLE ACCOUNTADMIN;

-- Step 1: a restricted role
CREATE ROLE IF NOT EXISTS RESTRICTED_ANALYST;

-- Step 2: the two things every caller needs
GRANT USE AI FUNCTIONS ON ACCOUNT           TO ROLE RESTRICTED_ANALYST;
GRANT DATABASE ROLE SNOWFLAKE.CORTEX_USER   TO ROLE RESTRICTED_ANALYST;

-- Step 3A: account-wide allowlist (affects EVERY role, not just this one)
ALTER ACCOUNT SET CORTEX_MODELS_ALLOWLIST = 'mistral-large3,llama3.1-8b';


In [ ]:
%%sql
--   Valid values: 'All' (default) | 'None' | comma-separated lowercase model names
--   From August 2026 the only change still permitted is setting it to 'None';
--   the parameter is removed later in 2026.

-- Step 3B: per-role model access, the mechanism that actually differentiates roles
CALL SNOWFLAKE.MODELS.CORTEX_BASE_MODELS_REFRESH();   -- populates SNOWFLAKE.MODELS; also runs daily


In [ ]:
%%sql
GRANT APPLICATION ROLE SNOWFLAKE."CORTEX-MODEL-ROLE-LLAMA3.1-8B" TO ROLE RESTRICTED_ANALYST;


In [ ]:
%%sql -r allowlist_param
-- Verify
SHOW PARAMETERS LIKE 'CORTEX_MODELS_ALLOWLIST' IN ACCOUNT;


In [ ]:
%%sql
SHOW CORTEX BASE MODELS IN SCHEMA SNOWFLAKE.MODELS;   -- output is filtered by the current role's grants


> ### ⚠️ Common misconceptions
>
> **"I set `CORTEX_MODELS_ALLOWLIST = 'None'`, so nobody can use any model now."**
> Only if nobody holds a model application role. Allowlist and model RBAC are combined with OR, so a
> role holding `CORTEX-MODEL-ROLE-LLAMA3.1-8B` keeps that model and the `'None'` setting changes
> nothing for them. The failure mode is silent: you believe the account is locked and inference keeps
> flowing.
> → [AISQL privileges and model access](https://docs.snowflake.com/en/user-guide/snowflake-cortex/aisql-privileges-and-access)
>
> **"Granting `CORTEX_USER` is enough to call an AI function."**
> It is half of the requirement. You also need the account privilege `USE AI FUNCTIONS` (or a
> per-function `USE AI FUNCTION <name>` grant). Miss either half and the call fails with an
> insufficient-privileges error on the function, which looks identical whichever half is missing.
> → [Snowflake Cortex AISQL](https://docs.snowflake.com/en/user-guide/snowflake-cortex/aisql)
>
> **"`AI_FUNCTIONS_USER` is just a smaller `CORTEX_USER`, so anything that works under one works under the other."**
> `AI_FUNCTIONS_USER` covers scalar AI functions only. `AI_AGG` and `AI_SUMMARIZE_AGG` are excluded,
> and so is every Cortex service. The symptom is a user who can run `AI_CLASSIFY` all day and gets a
> permissions error the moment a query uses `AI_AGG`.
> → [SNOWFLAKE database roles](https://docs.snowflake.com/en/sql-reference/snowflake-db-roles)
>
> **"A database role can be granted straight to a user."**
> It cannot. Grant the database role to one of your account roles, then grant that account role to
> the user. Trying it directly fails outright rather than silently, which is the friendlier outcome.
> → [SNOWFLAKE database roles](https://docs.snowflake.com/en/sql-reference/snowflake-db-roles)


In [ ]:
%%sql
-- Application Roles for Cortex Services
-- These are database roles in the SNOWFLAKE system database — not account roles

-- Grant Cortex Analyst access (for text-to-SQL features)
GRANT DATABASE ROLE SNOWFLAKE.CORTEX_ANALYST_USER TO ROLE RESTRICTED_ANALYST;

-- Grant Cortex Search access
GRANT DATABASE ROLE SNOWFLAKE.CORTEX_USER TO ROLE RESTRICTED_ANALYST;


In [ ]:
%%sql -r cortex_db_roles
-- Show all Cortex-related database roles
SHOW DATABASE ROLES IN DATABASE SNOWFLAKE;


In [ ]:
%%sql -r role_grants
-- Check what a role currently has
SHOW GRANTS TO ROLE RESTRICTED_ANALYST;


---
## Gate 5, part one: what the model is allowed to see, and to say

Two controls that sound alike and do opposite things.

**`AI_REDACT` acts on the input.** It finds PII in text and replaces it with typed placeholders such
as `[NAME]` and `[EMAIL]` before anything else touches the string. Signature:

```sql
AI_REDACT( <input> [, <categories ARRAY> ] [, <return_error_details> ] [, <mode> ] )
-- mode 'redact' (default) -> text with placeholders substituted
-- mode 'detect'           -> an OBJECT with a spans array: {category, start, end, text}
```

`categories` is an ARRAY of category names; omit it and every supported category is redacted.

**Cortex Guard acts on the output.** You switch it on per call with the `guardrails` key inside
`model_parameters`, and its documented job is to filter potentially unsafe and harmful responses from
the model. The default is `FALSE`.

```sql
AI_COMPLETE(<model>, <prompt> [, <model_parameters>, <response_format>, <show_details>])
-- model_parameters defaults: temperature 0, top_p 0, max_tokens 4096, guardrails FALSE
```

Separately and more recently, **Cortex AI Guardrails** is an account-level feature aimed at agents:
you enable it with `ALTER ACCOUNT SET AI_SETTINGS = …` using `advanced_prompt_injection`, it scans
each tool's output for indirect prompt injection, and its scans are metered by tokens and reported in
`SNOWFLAKE.ACCOUNT_USAGE.CORTEX_AI_GUARDRAILS_USAGE_HISTORY` (`GUARDRAILS_SIGNAL`,
`GUARDRAIL_RESULTS`, `TOKENS`, `TOKEN_CREDITS`).

Neither control makes an answer *true*. Safety and accuracy are different problems with different
fixes, and mixing them up is a reliable way to lose a mark.

→ [More on AI_REDACT](https://docs.snowflake.com/en/sql-reference/functions/ai_redact)
→ [More on Cortex AI Guardrails](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-ai-guardrails)


In [ ]:
%%sql -r guardrails_on
-- Cortex Guard filters potentially unsafe and harmful responses. Default is FALSE.
-- Enabled through the model_parameters object of AI_COMPLETE.
SELECT
    ticket_id,
    AI_COMPLETE(
        model  => 'llama3.1-8b',
        prompt => 'Respond to this customer complaint professionally: ' || ticket_text,
        model_parameters => {'guardrails': TRUE, 'temperature': 0}
    ) AS safe_response
FROM GENAI_STUDY.PUBLIC.SUPPORT_TICKETS
WHERE ticket_id = 1007;


In [ ]:
%%sql -r guardrails_details
-- show_details => TRUE returns choices, created, model and a usage object
-- with prompt_tokens, completion_tokens and total_tokens.
SELECT AI_COMPLETE(
    model  => 'llama3.1-8b',
    prompt => 'Summarize: ' || ticket_text,
    model_parameters => {'guardrails': TRUE},
    show_details     => TRUE
) AS detailed
FROM GENAI_STUDY.PUBLIC.SUPPORT_TICKETS WHERE ticket_id = 1007;

-- Guard filters the RESPONSE. Redacting the INPUT is a separate control (AI_REDACT).
-- Agent-level guardrail scans are metered separately and reported in
-- SNOWFLAKE.ACCOUNT_USAGE.CORTEX_AI_GUARDRAILS_USAGE_HISTORY.


In [ ]:
%%sql -r redact_then_complete
-- Data safety: AI_REDACT before AI processing
-- Best practice: redact PII BEFORE sending data to any AI function

-- Anti-pattern (PII sent to LLM)
-- SELECT AI_COMPLETE('llama3.1-8b', ticket_text) FROM SUPPORT_TICKETS;

-- Correct pattern (redact first)
WITH safe_inputs AS (
    SELECT
        ticket_id,
        AI_REDACT(ticket_text) AS clean_text  -- PII removed first
    FROM GENAI_STUDY.PUBLIC.SUPPORT_TICKETS
    WHERE status = 'open'
)
SELECT
    ticket_id,
    AI_COMPLETE('llama3.1-8b', 'Summarize this ticket: ' || clean_text) AS summary
FROM safe_inputs;

-- For highly sensitive data: also consider dynamic data masking policies
-- so raw PII is never visible to the analyst role even in SELECT statements


---
## Reaching Cortex from outside Snowflake

When the caller is an application rather than a person in a worksheet, the gates are the same — the
role still needs the account privilege, a database role and model access. Only the authentication
changes.

| Method | Use case | Token |
|---|---|---|
| Key-pair (JWT) | Server-to-server | RS256-signed JWT; `ALTER USER … SET RSA_PUBLIC_KEY = '…'` |
| OAuth | External apps, via Snowflake OAuth or an external IdP | Access token in `Authorization: Bearer …` |
| Session token | Inside Snowflake — notebooks, Streamlit, stored procedures | Supplied for you |

`SNOWFLAKE.CORTEX_REST_API_USER` is the database role that grants use of the Cortex REST API without
granting the rest of Cortex — the right choice for an application account whose only job is to call
the API.

```
1. Generate an RSA key pair
2. ALTER USER <u> SET RSA_PUBLIC_KEY = '<public key>';
3. Sign a JWT. iss = <ACCOUNT>.<USER>.SHA256:<fingerprint>, sub = <ACCOUNT>.<USER>,
   both uppercase; the JWT is valid for at most one hour whatever exp says.
4. POST with: Authorization: Bearer <JWT>
              X-Snowflake-Authorization-Token-Type: KEYPAIR_JWT   (optional)
              Content-Type: application/json
```

→ [More on Snowflake REST API authentication](https://docs.snowflake.com/en/developer-guide/snowflake-rest-api/authentication)

---
## Worked scenario: a GDPR-constrained analyst role

**The situation.** Analysts must be able to use Cortex, but only with one small model, all ticket
text must be free of PII before any AI call, and processing must stay in the EU.

```sql
USE ROLE ACCOUNTADMIN;

-- 1. Close the permissive defaults. Until you do this, every role design below is decorative.
REVOKE USE AI FUNCTIONS ON ACCOUNT FROM ROLE PUBLIC;
REVOKE DATABASE ROLE SNOWFLAKE.CORTEX_USER FROM ROLE PUBLIC;

-- 2. Give the role exactly the two things a Cortex call needs
CREATE ROLE IF NOT EXISTS GDPR_ANALYST;
GRANT USE AI FUNCTIONS ON ACCOUNT         TO ROLE GDPR_ANALYST;
GRANT DATABASE ROLE SNOWFLAKE.CORTEX_USER TO ROLE GDPR_ANALYST;

-- 3. Restrict WHICH model. Per-role control is model RBAC; the allowlist is account-wide.
ALTER ACCOUNT SET CORTEX_MODELS_ALLOWLIST = 'None';   -- switch the account-wide path off
CALL SNOWFLAKE.MODELS.CORTEX_BASE_MODELS_REFRESH();
GRANT APPLICATION ROLE SNOWFLAKE."CORTEX-MODEL-ROLE-LLAMA3.1-8B" TO ROLE GDPR_ANALYST;

-- 4. Redact at the view layer so analysts never hold raw PII
CREATE OR REPLACE SECURE VIEW GENAI_STUDY.PUBLIC.TICKETS_GDPR AS
SELECT
    ticket_id, created_at, category, status, priority, satisfaction_score,
    AI_REDACT(ticket_text) AS ticket_text
FROM GENAI_STUDY.PUBLIC.SUPPORT_TICKETS;

-- 5. Grant the view, never the base table
GRANT USAGE  ON DATABASE GENAI_STUDY        TO ROLE GDPR_ANALYST;
GRANT USAGE  ON SCHEMA   GENAI_STUDY.PUBLIC TO ROLE GDPR_ANALYST;
GRANT SELECT ON VIEW GENAI_STUDY.PUBLIC.TICKETS_GDPR TO ROLE GDPR_ANALYST;

-- 6. Keep inference in the EU
ALTER ACCOUNT SET CORTEX_ENABLED_CROSS_REGION = 'AWS_EU';   -- or 'DISABLED' for home region only
```

**What step 6 costs you:** any model not served in AWS Europe stops being available to the whole
account, not just to this role. Residency is an account-wide decision.


> ### 🤔 Stop and think
>
> - Your account has three teams with three different model budgets. Model RBAC gives you per-role
>   control but every new model needs a new grant to every role that should have it. The allowlist is
>   one line but identical for everyone — and it is being retired. What does the maintenance burden of
>   per-role grants actually cost your team each quarter, and who owns that work?
> - `ANY_REGION` maximises model availability; `AWS_EU` maximises the strength of your residency
>   claim. If your legal team cannot tell you which one they need, what evidence would you gather
>   before choosing — and what would you have to undo if you chose wrong?
> - Redacting PII at the view layer means analysts never see real names. It also means an analyst
>   debugging a bad answer cannot tell whether the model failed or the redaction ate something it
>   needed. Where would you draw that line, and how would you review it?


In [ ]:
%%sql
-- ============================================================
-- CORTEX AI ACCESS-CONTROL REFERENCE
-- ============================================================
USE ROLE ACCOUNTADMIN;

-- 1. ACCOUNT PRIVILEGE: USE AI FUNCTIONS  (granted to PUBLIC by default)
REVOKE USE AI FUNCTIONS ON ACCOUNT FROM ROLE PUBLIC;
GRANT  USE AI FUNCTIONS ON ACCOUNT TO ROLE RESTRICTED_ANALYST;

-- 2. PER-FUNCTION PRIVILEGE (OR relationship with the blanket privilege above)
GRANT USE AI FUNCTION AI_COMPLETE       ON ACCOUNT TO ROLE RESTRICTED_ANALYST;
GRANT USE AI FUNCTION AI_CLASSIFY       ON ACCOUNT TO ROLE RESTRICTED_ANALYST;
GRANT USE AI FUNCTION AI_EXTRACT        ON ACCOUNT TO ROLE RESTRICTED_ANALYST;
GRANT USE AI FUNCTION AI_FILTER         ON ACCOUNT TO ROLE RESTRICTED_ANALYST;
GRANT USE AI FUNCTION AI_EMBED          ON ACCOUNT TO ROLE RESTRICTED_ANALYST;
GRANT USE AI FUNCTION AI_SENTIMENT      ON ACCOUNT TO ROLE RESTRICTED_ANALYST;
GRANT USE AI FUNCTION AI_TRANSLATE      ON ACCOUNT TO ROLE RESTRICTED_ANALYST;
GRANT USE AI FUNCTION AI_REDACT         ON ACCOUNT TO ROLE RESTRICTED_ANALYST;
GRANT USE AI FUNCTION AI_PARSE_DOCUMENT ON ACCOUNT TO ROLE RESTRICTED_ANALYST;
GRANT USE AI FUNCTION AI_TRANSCRIBE     ON ACCOUNT TO ROLE RESTRICTED_ANALYST;
GRANT USE AI FUNCTION AI_COUNT_TOKENS   ON ACCOUNT TO ROLE RESTRICTED_ANALYST;

-- 3. DATABASE ROLES (in the SNOWFLAKE database)
REVOKE DATABASE ROLE SNOWFLAKE.CORTEX_USER FROM ROLE PUBLIC;                     -- PUBLIC by default
GRANT  DATABASE ROLE SNOWFLAKE.CORTEX_USER          TO ROLE RESTRICTED_ANALYST;
GRANT  DATABASE ROLE SNOWFLAKE.AI_FUNCTIONS_USER    TO ROLE RESTRICTED_ANALYST;  -- scalar only
GRANT  DATABASE ROLE SNOWFLAKE.CORTEX_EMBED_USER    TO ROLE RESTRICTED_ANALYST;  -- embeddings
GRANT  DATABASE ROLE SNOWFLAKE.CORTEX_ANALYST_USER  TO ROLE RESTRICTED_ANALYST;  -- Cortex Analyst
GRANT  DATABASE ROLE SNOWFLAKE.CORTEX_AGENT_USER    TO ROLE RESTRICTED_ANALYST;  -- Cortex Agents
GRANT  DATABASE ROLE SNOWFLAKE.CORTEX_REST_API_USER TO ROLE RESTRICTED_ANALYST;  -- REST API only


In [ ]:
%%sql -r snowflake_db_roles
SHOW DATABASE ROLES IN DATABASE SNOWFLAKE;


In [ ]:
%%sql -r base_models_refresh
-- 4. MODEL RBAC — per-model application roles
CALL SNOWFLAKE.MODELS.CORTEX_BASE_MODELS_REFRESH();


In [ ]:
%%sql -r base_models
SHOW CORTEX BASE MODELS IN SCHEMA SNOWFLAKE.MODELS;


In [ ]:
%%sql -r app_roles
SHOW APPLICATION ROLES IN APPLICATION SNOWFLAKE;


In [ ]:
%%sql
GRANT APPLICATION ROLE SNOWFLAKE."CORTEX-MODEL-ROLE-LLAMA3.3-70B" TO ROLE RESTRICTED_ANALYST;
GRANT APPLICATION ROLE SNOWFLAKE."CORTEX-MODEL-ROLE-ALL"          TO ROLE RESTRICTED_ANALYST;

-- 5. ACCOUNT-WIDE ALLOWLIST
ALTER ACCOUNT SET CORTEX_MODELS_ALLOWLIST = 'mistral-large3,llama3.3-70b';
-- ALTER ACCOUNT SET CORTEX_MODELS_ALLOWLIST = 'None';   -- rely purely on model RBAC
-- ALTER ACCOUNT SET CORTEX_MODELS_ALLOWLIST = 'All';    -- the default

-- ============================================================
-- | Control                                | Scope                              | Default    |
-- |----------------------------------------|------------------------------------|------------|
-- | USE AI FUNCTIONS (account priv)        | all AI functions                   | PUBLIC     |
-- | USE AI FUNCTION <name> (per function)  | one AI function                    | none       |
-- | SNOWFLAKE.CORTEX_USER (db role)        | all AI functions + Cortex services | PUBLIC     |
-- | SNOWFLAKE.AI_FUNCTIONS_USER (db role)  | scalar AI funcs; NO AI_AGG /       | not PUBLIC |
-- |                                        | AI_SUMMARIZE_AGG; no services      |            |
-- | SNOWFLAKE.CORTEX_EMBED_USER (db role)  | AI_EMBED, EMBED_TEXT_768/1024      | not PUBLIC |
-- | SNOWFLAKE.CORTEX_ANALYST_USER          | Cortex Analyst only                | not PUBLIC |
-- | SNOWFLAKE.CORTEX_AGENT_USER            | Cortex Agents API only             | not PUBLIC |
-- | SNOWFLAKE.CORTEX_REST_API_USER         | Cortex REST API only               | not PUBLIC |
-- | SNOWFLAKE.COPILOT_USER                 | Cortex Code in Snowsight           | PUBLIC     |
-- | CORTEX-MODEL-ROLE-<MODEL> (app role)   | one model                          | none       |
-- | CORTEX-MODEL-ROLE-ALL (app role)       | every model                        | none       |
-- | CORTEX_MODELS_ALLOWLIST (parameter)    | account-wide model list            | 'All'      |
-- ============================================================


---
## Working through the OR

Four configurations, and what a role actually gets under each.

| `CORTEX_MODELS_ALLOWLIST` | Application role granted | Result for that role |
|---|---|---|
| `'mistral-large2'` | none | `mistral-large2` only — and the same for every other role |
| `'None'` | `CORTEX-MODEL-ROLE-LLAMA3.1-70B` | `llama3.1-70b` only, and only for this role |
| `'mistral-large2'` | `CORTEX-MODEL-ROLE-LLAMA3.1-70B` | both models — the OR |
| `'None'` | none | no models |

The resolution order in words: if the model object exists in `SNOWFLAKE.MODELS` and the calling role
holds its application role, access is granted. Otherwise the model name is checked against
`CORTEX_MODELS_ALLOWLIST`. Only when both checks fail is the call denied.

`SHOW CORTEX BASE MODELS IN SCHEMA SNOWFLAKE.MODELS` is the verification tool here: its output is
filtered by the current role's model grants, so running it as the restricted role tells you what that
role can really reach. Its columns are `created_on`, `name`, `model_type`, `database_name`,
`schema_name`, `owner`, `lifecycle_status`, `in_region_availability`, `legacy_date`, `eol_date` and
`cross_region_availability`.

→ [More on SHOW CORTEX BASE MODELS](https://docs.snowflake.com/en/sql-reference/sql/show-cortex-base-models)


In [ ]:
%%sql
-- ============================================================
-- CASE 1: Pure RBAC — Allowlist set to 'None', use app roles
-- ============================================================
-- Best practice for fine-grained per-role model control

USE ROLE ACCOUNTADMIN;

-- Disable the allowlist entirely — rely exclusively on Model RBAC
ALTER ACCOUNT SET CORTEX_MODELS_ALLOWLIST = 'None';


In [ ]:
%%sql -r refresh_for_rbac
-- Refresh model objects (creates app roles in SNOWFLAKE.MODELS)
CALL SNOWFLAKE.MODELS.CORTEX_BASE_MODELS_REFRESH();


In [ ]:
%%sql
-- Grant specific model access to a role
GRANT APPLICATION ROLE SNOWFLAKE."CORTEX-MODEL-ROLE-LLAMA3.1-70B" TO ROLE RESTRICTED_ANALYST;
GRANT APPLICATION ROLE SNOWFLAKE."CORTEX-MODEL-ROLE-MISTRAL-LARGE2" TO ROLE RESTRICTED_ANALYST;

-- Succeeds: RBAC grants access even though the allowlist is 'None'
-- USE ROLE RESTRICTED_ANALYST;
-- SELECT AI_COMPLETE('llama3.1-70b', 'Hello');

-- Fails: no application role granted for this model, and the allowlist is 'None'
-- SELECT AI_COMPLETE('claude-sonnet-4-6', 'Hello');


In [ ]:
%%sql
-- ============================================================
-- CASE 2: Allowlist ONLY — No RBAC, broad account-wide control
-- ============================================================
-- Simpler but less granular — same models available to ALL roles

USE ROLE ACCOUNTADMIN;

-- Allow only specific models account-wide (applies to everyone)
ALTER ACCOUNT SET CORTEX_MODELS_ALLOWLIST = 'mistral-large3,llama3.3-70b,llama3.1-8b';

-- No need for CORTEX_BASE_MODELS_REFRESH or app role grants
-- Any role with CORTEX_USER + USE AI FUNCTIONS can use these 3 models

-- Test: This SUCCEEDS (model is in allowlist)
-- SELECT AI_COMPLETE('mistral-large2', 'Hello');

-- Test: This FAILS (model NOT in allowlist, no RBAC fallback)
-- SELECT AI_COMPLETE('llama3.1-405b', 'Hello');  -- ❌ Not in allowlist


In [ ]:
%%sql
-- ============================================================
-- CASE 3: Combined — Allowlist + RBAC together (OR logic)
-- ============================================================
-- Allowlist provides baseline, RBAC grants additional per-role access

USE ROLE ACCOUNTADMIN;

-- Allowlist: everyone gets mistral-large2
ALTER ACCOUNT SET CORTEX_MODELS_ALLOWLIST = 'mistral-large2';


In [ ]:
%%sql -r r3_1_34
-- RBAC: grant additional model access to specific roles
CALL SNOWFLAKE.MODELS.CORTEX_BASE_MODELS_REFRESH();


In [ ]:
%%sql
GRANT APPLICATION ROLE SNOWFLAKE."CORTEX-MODEL-ROLE-LLAMA3.1-70B" TO ROLE RESTRICTED_ANALYST;

-- Result for RESTRICTED_ANALYST:
--   mistral-large2  → ✅ (via allowlist)
--   llama3.1-70b    → ✅ (via RBAC app role)
--   claude-sonnet-4-6   → ❌ (not in allowlist AND no app role)

-- Result for OTHER roles (no app roles granted):
--   mistral-large2  → ✅ (via allowlist)
--   llama3.1-70b    → ❌ (not in allowlist, no RBAC)
--   claude-sonnet-4-6   → ❌ (not in allowlist, no RBAC)


In [ ]:
%%sql
-- ============================================================
-- CASE 4: Differentiated access — different models per role
-- ============================================================
-- Most secure: each team gets only the models they need

USE ROLE ACCOUNTADMIN;

-- Disable allowlist to enforce pure RBAC
ALTER ACCOUNT SET CORTEX_MODELS_ALLOWLIST = 'None';


In [ ]:
%%sql -r r3_1_37
CALL SNOWFLAKE.MODELS.CORTEX_BASE_MODELS_REFRESH();


In [ ]:
%%sql
-- Data Science team: gets large expensive models
CREATE ROLE IF NOT EXISTS DATA_SCIENCE_ROLE;
GRANT DATABASE ROLE SNOWFLAKE.CORTEX_USER TO ROLE DATA_SCIENCE_ROLE;
GRANT USE AI FUNCTIONS ON ACCOUNT TO ROLE DATA_SCIENCE_ROLE;
GRANT APPLICATION ROLE SNOWFLAKE."CORTEX-MODEL-ROLE-LLAMA3.1-405B" TO ROLE DATA_SCIENCE_ROLE;
GRANT APPLICATION ROLE SNOWFLAKE."CORTEX-MODEL-ROLE-MISTRAL-LARGE2" TO ROLE DATA_SCIENCE_ROLE;

-- Analyst team: gets only small/cheap models
CREATE ROLE IF NOT EXISTS ANALYST_ROLE;
GRANT DATABASE ROLE SNOWFLAKE.CORTEX_USER TO ROLE ANALYST_ROLE;
GRANT USE AI FUNCTIONS ON ACCOUNT TO ROLE ANALYST_ROLE;
GRANT APPLICATION ROLE SNOWFLAKE."CORTEX-MODEL-ROLE-MISTRAL-7B" TO ROLE ANALYST_ROLE;

-- Embedded app: gets one specific model only
CREATE ROLE IF NOT EXISTS EMBEDDED_APP_ROLE;
GRANT DATABASE ROLE SNOWFLAKE.AI_FUNCTIONS_USER TO ROLE EMBEDDED_APP_ROLE;
GRANT USE AI FUNCTION AI_COMPLETE ON ACCOUNT TO ROLE EMBEDDED_APP_ROLE;
GRANT APPLICATION ROLE SNOWFLAKE."CORTEX-MODEL-ROLE-MISTRAL-7B" TO ROLE EMBEDDED_APP_ROLE;


In [ ]:
%%sql -r r3_1_39
-- Verify grants
SHOW GRANTS TO ROLE DATA_SCIENCE_ROLE;


In [ ]:
%%sql -r r3_1_40
SHOW GRANTS TO ROLE ANALYST_ROLE;


In [ ]:
%%sql -r r3_1_41
SHOW GRANTS TO ROLE EMBEDDED_APP_ROLE;


In [ ]:
%%sql
-- ============================================================
-- Secondary roles hide permission problems
-- ============================================================
-- With ACCOUNTADMIN active as a secondary role, every model looks accessible.

USE ROLE ACCOUNTADMIN;

-- Turn secondary roles off so the test reflects one role only
USE SECONDARY ROLES NONE;

USE ROLE RESTRICTED_ANALYST;

-- Now this reflects ONLY what RESTRICTED_ANALYST can reach
-- SELECT AI_COMPLETE('llama3.1-70b', 'Test access');

-- Put them back when you are done
-- USE SECONDARY ROLES ALL;


---
## Reducing hallucination and bias is a governance job

Data safety in this domain is not only PII. Making answers trustworthy is a control problem, not a
prompt trick, and the controls sit in different places.

| Control | Mechanism | What it actually fixes |
|---|---|---|
| Ground the answer | Retrieve with Cortex Search, instruct the model to answer only from that context | Invented facts — the dominant failure mode |
| `temperature: 0` | `model_parameters => {'temperature': 0}`, already the default for `AI_COMPLETE` | Run-to-run variability |
| Structured output | `response_format` with a JSON schema, or `AI_CLASSIFY` / `AI_EXTRACT` | Free-text drift and unparseable answers |
| Refusal instruction | "If the answer is not in the context, say you do not know." | Confident answers to unanswerable questions |
| `guardrails: TRUE` | Cortex Guard filters the response | Unsafe or harmful output — **not** factual accuracy |
| Fine-tuning | Train on your own labelled examples | Consistent failures on domain vocabulary |
| Measure it | AI Observability **groundedness** and **correctness** metrics | Whether any of the above worked |

**Bias has the same shape.** Constrain the input (redact attributes the model should not weigh),
constrain the output (a fixed label set through `AI_CLASSIFY` rather than free text), and measure
against an evaluation set that contains the cases you are worried about. Cortex Guard is a safety
filter, not a fairness control, and no parameter makes a model fair.

> **How the exam words it:** if the complaint is that answers are *made up*, the metric is
> **groundedness** and the first fix is grounding plus a refusal instruction. If the complaint is that
> answers are *unsafe*, that is Cortex Guard. The two are routinely swapped in distractors.

→ [More on AI Observability metrics](https://docs.snowflake.com/en/user-guide/snowflake-cortex/ai-observability/reference)


In [ ]:
%%sql -r r3_1_44
-- Anti-hallucination pattern: ground, constrain, and refuse
SELECT AI_COMPLETE(
    model  => 'llama3.1-8b',
    prompt => 'Answer ONLY from the context below. If the answer is not in the context, reply '
           || 'exactly "I do not know".' || CHR(10)
           || 'Context: ' || retrieved_context || CHR(10)
           || 'Question: ' || user_question,
    model_parameters => {'temperature': 0, 'guardrails': TRUE},
    response_format  => {
        'type': 'json',
        'schema': {
            'type': 'object',
            'properties': {'answer': {'type':'string'}, 'supported_by_context': {'type':'boolean'}},
            'required': ['answer', 'supported_by_context']
        }
    }
) AS grounded_answer
FROM GENAI_STUDY.PUBLIC.RAG_REQUESTS;

---
## Turning AI features off

Two categories, and the distinction is the exam point.

### On by default → revoke from `PUBLIC`
```sql
REVOKE DATABASE ROLE SNOWFLAKE.CORTEX_USER  FROM ROLE PUBLIC;   -- Cortex AI functions and services
REVOKE DATABASE ROLE SNOWFLAKE.COPILOT_USER FROM ROLE PUBLIC;   -- Cortex Code in Snowsight
REVOKE USE AI FUNCTIONS ON ACCOUNT          FROM ROLE PUBLIC;   -- the account privilege
```
Then grant back to named roles.

### Opt-in or separately controlled
| Feature | How to turn it off |
|---|---|
| Cortex Analyst | `ALTER ACCOUNT SET ENABLE_CORTEX_ANALYST = FALSE;` (the parameter defaults to `TRUE`) |
| Cortex embedding functions | `REVOKE DATABASE ROLE SNOWFLAKE.CORTEX_EMBED_USER FROM ROLE <r>;` |
| Fine-tuning | `REVOKE CREATE MODEL ON SCHEMA <s> FROM ROLE <r>;` |
| Provisioned Throughput | Revoke the account-level `CREATE PROVISIONED THROUGHPUT` privilege — `ACCOUNTADMIN` holds it by default |
| Model access | `ALTER ACCOUNT SET CORTEX_MODELS_ALLOWLIST = 'None';` and grant no model application roles |
| Region routing | `ALTER ACCOUNT SET CORTEX_ENABLED_CROSS_REGION = 'DISABLED';` |

### The `ACCOUNTADMIN` ceiling
`ACCOUNTADMIN` has complete access to every feature in the account, AI features included. Revoking
database roles from `PUBLIC` does not restrict it.

An account parameter such as `ENABLE_CORTEX_ANALYST = FALSE` does bind everyone including
`ACCOUNTADMIN` — but `ACCOUNTADMIN` can always set it back to `TRUE`.

> Role revocation controls other people. Parameters control the account. Neither controls
> `ACCOUNTADMIN`, because that role can undo either. The real control is not handing the role out,
> backed by monitoring in `ACCOUNT_USAGE`.

→ [More on opting out of Snowflake AI features](https://docs.snowflake.com/en/user-guide/snowflake-cortex/opting-out)


In [ ]:
%%sql -r r3_1_46
-- Audit the opt-out posture: what is PUBLIC still carrying?
SHOW GRANTS TO ROLE PUBLIC;

In [ ]:
%%sql -r r3_1_47
-- Detect AI features being switched back on — alert on new rows here
SELECT
    DATE_TRUNC('day', START_TIME) AS day,
    SERVICE_TYPE,
    SUM(CREDITS_USED)             AS credits
FROM SNOWFLAKE.ACCOUNT_USAGE.METERING_HISTORY
WHERE SERVICE_TYPE = 'AI_SERVICES'
  AND START_TIME >= DATEADD('day', -30, CURRENT_TIMESTAMP())
GROUP BY 1, 2
ORDER BY day DESC;

> ### ⚠️ Common misconceptions
>
> **"I revoked `CORTEX_USER` from `PUBLIC`, so the account is locked down."**
> `ACCOUNTADMIN` still has complete access to every AI feature regardless of what `PUBLIC` holds, and
> so does anyone you granted `ACCOUNTADMIN` to. The revoke is real, but it never covered that role in
> the first place — so the audit finding survives your fix.
> → [Opting out of Snowflake AI features](https://docs.snowflake.com/en/user-guide/snowflake-cortex/opting-out)
>
> **"Cortex Guard will stop the model making things up."**
> Guard filters unsafe and harmful responses. A calm, polite, entirely invented answer sails straight
> through it. Hallucination is fixed by grounding and a refusal instruction, and measured by the
> groundedness metric.
> → [AI_COMPLETE (single string)](https://docs.snowflake.com/en/sql-reference/functions/ai_complete-single-string)
>
> **"My restricted role can see every model, so the grants did not work."**
> More often your restricted role is fine and `ACCOUNTADMIN` is active as a *secondary* role, which
> silently supplies the missing access. `USE SECONDARY ROLES NONE;` before you test, and
> `USE SECONDARY ROLES ALL;` afterwards. Skipping this produces a test that passes when it should fail.
> → [AISQL privileges and model access](https://docs.snowflake.com/en/user-guide/snowflake-cortex/aisql-privileges-and-access)


---

## Check your understanding

Twelve questions on this notebook. Answer before expanding.

**1.** Which two grants must a role hold before any Cortex AI function call will succeed?

<details><summary>Show answer</summary>

The account privilege `USE AI FUNCTIONS ON ACCOUNT` (or a per-function `USE AI FUNCTION <name>` grant)
**and** one of the Cortex database roles — `SNOWFLAKE.CORTEX_USER` or `SNOWFLAKE.AI_FUNCTIONS_USER`.
They are two separate gates, and holding only one produces an insufficient-privileges error that looks
the same either way. Tempting wrong answer: "just `CORTEX_USER`" — that was sufficient before the
account privilege existed and is now half the requirement.

→ [Snowflake Cortex AISQL](https://docs.snowflake.com/en/user-guide/snowflake-cortex/aisql)

</details>

**2.** What is the default value of `CORTEX_MODELS_ALLOWLIST`, and at what level can it be set?

<details><summary>Show answer</summary>

The default is `'All'`, and it can be set only at the account level — not per user and not per session.
Its other valid values are `'None'` and a comma-separated list of lowercase model names. That
account-only scope is precisely why it cannot give two teams different model sets.

→ [AISQL privileges and model access](https://docs.snowflake.com/en/user-guide/snowflake-cortex/aisql-privileges-and-access)

</details>

**3.** Which role can change `CORTEX_ENABLED_CROSS_REGION`, and what does `DISABLED` mean?

<details><summary>Show answer</summary>

Only `ACCOUNTADMIN`, using `ALTER ACCOUNT`. `DISABLED` means no cross-region routing at all: requests
are served only by models available in your own region, and a call for a model that is not there
fails rather than being routed. `ORGADMIN` is a tempting distractor — it manages organization-level
objects, not this parameter.

→ [Cross-region inference](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cross-region-inference)

</details>

**4.** An administrator runs `ALTER ACCOUNT SET CORTEX_MODELS_ALLOWLIST = 'None';` and reports the
account locked down. A role still runs `AI_COMPLETE('llama3.3-70b', …)` successfully. What happened?

<details><summary>Show answer</summary>

That role holds `SNOWFLAKE."CORTEX-MODEL-ROLE-LLAMA3.3-70B"`. The allowlist and model RBAC are combined
with OR, so an application role grants access on its own and the allowlist setting is simply not
consulted for it. `'None'` turns the allowlist path off; it does not revoke anything.

→ [AISQL privileges and model access](https://docs.snowflake.com/en/user-guide/snowflake-cortex/aisql-privileges-and-access)

</details>

**5.** A role holds `USE AI FUNCTIONS` and `SNOWFLAKE.AI_FUNCTIONS_USER`. `AI_CLASSIFY` works;
`AI_AGG` fails with a permissions error. Why, and what is the fix?

<details><summary>Show answer</summary>

`AI_FUNCTIONS_USER` covers scalar AI functions only and explicitly excludes `AI_AGG` and
`AI_SUMMARIZE_AGG`. Grant `SNOWFLAKE.CORTEX_USER` instead if the role genuinely needs aggregates.
Re-granting the account privilege changes nothing — gate 1 already passed, which is why the error is
so confusing the first time you see it.

→ [SNOWFLAKE database roles](https://docs.snowflake.com/en/sql-reference/snowflake-db-roles)

</details>

**6.** You grant a new role `CORTEX_USER`, `USE AI FUNCTIONS` and
`SNOWFLAKE."CORTEX-MODEL-ROLE-MISTRAL-LARGE2"`, but the grant fails saying the application role does
not exist. What is missing?

<details><summary>Show answer</summary>

The model objects have not been created in `SNOWFLAKE.MODELS` yet, so the per-model application roles
do not exist. Run `CALL SNOWFLAKE.MODELS.CORTEX_BASE_MODELS_REFRESH();` as `ACCOUNTADMIN`. Snowflake
also runs this daily, so waiting works too — forcing it is just faster.

→ [AISQL privileges and model access](https://docs.snowflake.com/en/user-guide/snowflake-cortex/aisql-privileges-and-access)

</details>

**7.** You test `RESTRICTED_ANALYST` and every model appears in `SHOW CORTEX BASE MODELS`. The grants
look correct. What did you forget?

<details><summary>Show answer</summary>

`USE SECONDARY ROLES NONE;`. With secondary roles active, a privileged role you also hold — commonly
`ACCOUNTADMIN` — contributes its access to the session, so the output reflects the union, not the role
you meant to test. The output of `SHOW CORTEX BASE MODELS` is filtered by the current role's model
grants, which makes it a good verification tool only once secondary roles are off.

→ [SHOW CORTEX BASE MODELS](https://docs.snowflake.com/en/sql-reference/sql/show-cortex-base-models)

</details>

**8.** A pipeline calls `AI_COMPLETE` on raw ticket text with `guardrails: TRUE` and the compliance
team objects. What is the actual gap?

<details><summary>Show answer</summary>

Guard filters the model's **response**. The customer's name, email and phone number in the prompt
still reached the model, because nothing redacted the input. Wrap the column in `AI_REDACT` — or better,
expose a secure view whose `ticket_text` is already redacted, so the raw column is never granted at all.

→ [AI_REDACT](https://docs.snowflake.com/en/sql-reference/functions/ai_redact)

</details>

**9.** `AI_REDACT(ticket_text, NULL, FALSE, 'detect')` — what does this return, and when is it the
right call to make?

<details><summary>Show answer</summary>

An OBJECT with a `spans` array, each entry carrying `category`, `start`, `end` and `text` — the PII it
found, with the original text left intact. Use it to audit what PII exists in a column before you
decide on a redaction policy. `'redact'` is the default mode and returns the rewritten string instead.

→ [AI_REDACT](https://docs.snowflake.com/en/sql-reference/functions/ai_redact)

</details>

**10.** Your legal team requires EU-only processing. Compare pinning
`CORTEX_ENABLED_CROSS_REGION = 'AWS_EU'` against `'DISABLED'`. What does each cost you?

<details><summary>Show answer</summary>

`'AWS_EU'` keeps routing available across AWS Europe, so you retain models that are absent from your
own region while staying inside the geography. `'DISABLED'` is stricter — home region only — and every
model not served there simply becomes unavailable, breaking pipelines that depend on it. Both are
account-wide: there is no way to give one role a different routing policy, and no single-region value.

→ [Cross-region inference](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cross-region-inference)

</details>

**11.** You need three teams on three different model sets, and you want the design to survive into
2027. Which mechanism do you build on, and what does it cost?

<details><summary>Show answer</summary>

Model RBAC: set `CORTEX_MODELS_ALLOWLIST = 'None'` to take the account-wide path out of play, then
grant per-model application roles per team. The cost is ongoing maintenance — every new model needs a
deliberate grant to each team that should have it, and nothing happens automatically. The allowlist is
one line and cannot differentiate teams, and from August 2026 the only change still permitted to it is
setting it to `'None'`, with removal later in 2026.

→ [AISQL privileges and model access](https://docs.snowflake.com/en/user-guide/snowflake-cortex/aisql-privileges-and-access)

</details>

**12.** Connecting to cost governance: after locking model access down, you want to prove that AI
spend actually fell. Which `ACCOUNT_USAGE` view answers "what did AI cost us each day", and which
column do you filter on?

<details><summary>Show answer</summary>

`SNOWFLAKE.ACCOUNT_USAGE.METERING_DAILY_HISTORY`, filtered on `SERVICE_TYPE = 'AI_SERVICES'` — the
value that covers Cortex AI Functions and Cortex Analyst. `METERING_HISTORY` is the same family at
hourly grain, and `QUERY_HISTORY` is a distractor: it has no credit column and no `SERVICE_TYPE`.
Notebook 3.3 takes this apart view by view.

→ [METERING_DAILY_HISTORY](https://docs.snowflake.com/en/sql-reference/account-usage/metering_daily_history)

</details>
